# Non-Contrastive and Masked SSL: BYOL, DINO, VICReg, MAE, and JEPA

This notebook explores modern self-supervised learning methods that avoid contrastive negative pairs:
1. **BYOL / SimSiam**: Asymmetric prediction architectures with stop-gradient dynamics.
2. **VICReg**: Variance-Invariance-Covariance regularization directly on embedding distributions.
3. **DINO & KoLeo**: Centered and sharpened self-distillation with differential entropy regularization.
4. **MAE & I-JEPA**: Masked autoencoding on visible patches and latent feature space prediction.

**Decision Boundary (When NOT to Use)**:
- Do not use non-contrastive methods if batch sizes are extremely small ($B < 8$) without batch-independent normalization or when multi-modal cross-attention alignment (CLIP) is strictly required.
- Do not use MAE for downstream tasks requiring dense high-frequency pixel prediction unless paired with hierarchical multi-scale decoders.

**Topic Bridges & Connects To**:
- `01_contrastive_ssl_simclr_moco_and_clip_siglip` (Precursor contrastive paradigm)
- `03_semi_supervised_fixmatch_mean_teacher_and_nnpu_learning` (Semi-supervised consistency targets)

## 2. Mathematical Formulations & Theoretical Foundations

- **BYOL / SimSiam Normalized Prediction Loss**:
  $$\mathcal{L}_{\text{BYOL}} = 2 - 2 \frac{p \cdot z}{\|p\|_2 \|z\|_2}, \quad \mathcal{O}(B \cdot D)$$
- **VICReg Multi-Objective Regularization**:
  $$\mathcal{L}_{\text{VICReg}} = \lambda s(Z, Z') + \mu [v(Z) + v(Z')] + \nu [c(Z) + c(Z')], \quad \mathcal{O}(B \cdot D^2)$$
- **DINO Centered Cross-Entropy Distillation**:
  $$\mathcal{L}_{\text{DINO}} = - \sum_k P_t(k) \log P_s(k), \quad \mathcal{O}(B \cdot K)$$
- **KoLeo Nearest-Neighbor Differential Entropy**:
  $$\mathcal{L}_{\text{KoLeo}} = - \frac{1}{B} \sum_{i=1}^B \log \left( \min_{j \neq i} \|z_i - z_j\|_2 + \epsilon \right), \quad \mathcal{O}(B^2 \cdot D)$$
- **I-JEPA Latent Representation MSE**:
  $$\mathcal{L}_{\text{I-JEPA}} = \frac{1}{M \cdot D} \sum_{m=1}^M \|\hat{s}_y^{(m)} - s_y^{(m)}\|_2^2, \quad \mathcal{O}(M \cdot D)$$

In [ ]:
import math
from pathlib import Path
import torch
import torch.nn.functional as F
from safetensors.torch import load_file

torch.manual_seed(42)
REPO_ROOT = next(p for p in Path.cwd().parents if (p / "Cargo.toml").exists()) if not (Path.cwd() / "Cargo.toml").exists() else Path.cwd()
GOLDEN_PATH = REPO_ROOT / "weights" / "08_learning_paradigms_rl_and_alignment" / "01_self_semi_meta_continual_federated" / "02_non_contrastive_and_masked_ssl_byol_dino_vicreg_mae_jepa.safetensors"
golden = load_file(str(GOLDEN_PATH))
print(f"Loaded {len(golden)} tensors from golden checkpoint.")

In [ ]:
# 1. BYOL Normalized Cosine Prediction Loss
def byol_prediction_loss(p: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
    p_norm = p / (torch.norm(p, dim=-1, keepdim=True) + 1e-12)
    z_norm = z / (torch.norm(z, dim=-1, keepdim=True) + 1e-12)
    cos_sim = (p_norm * z_norm).sum(dim=-1)
    return (2.0 - 2.0 * cos_sim).mean().unsqueeze(0)

byol_calc = byol_prediction_loss(golden["byol_p"], golden["byol_z"])
diff_byol = torch.abs(byol_calc - golden["byol_loss_ref"]).item()
print(f"BYOL Prediction Loss max abs error: {diff_byol:.2e}")
assert diff_byol < 1e-5

In [ ]:
# 2. VICReg Loss (Invariance + Variance + Covariance)
def vicreg_loss(
    za: torch.Tensor,
    zb: torch.Tensor,
    sim_coeff: float = 1.0,
    std_coeff: float = 1.0,
    cov_coeff: float = 0.04,
    gamma: float = 1.0,
    eps: float = 1e-4,
) -> torch.Tensor:
    b, d = za.shape
    sim_loss = F.mse_loss(za, zb)

    def var_loss(z: torch.Tensor) -> torch.Tensor:
        std = torch.sqrt(torch.var(z, dim=0, unbiased=False) + eps)
        return torch.mean(F.relu(gamma - std))

    std_loss = var_loss(za) + var_loss(zb)

    def cov_loss(z: torch.Tensor) -> torch.Tensor:
        z_c = z - z.mean(dim=0, keepdim=True)
        cov = (z_c.T @ z_c) / (b - 1)
        cov_off = cov - torch.diag(torch.diag(cov))
        return (cov_off ** 2).sum() / d

    cov_loss_val = cov_loss(za) + cov_loss(zb)
    return (sim_coeff * sim_loss + std_coeff * std_loss + cov_coeff * cov_loss_val).unsqueeze(0)

vicreg_calc = vicreg_loss(golden["vicreg_za"], golden["vicreg_zb"])
diff_vicreg = torch.abs(vicreg_calc - golden["vicreg_loss_ref"]).item()
print(f"VICReg Loss max abs error: {diff_vicreg:.2e}")
assert diff_vicreg < 1e-5

In [ ]:
# 3. DINO Distillation & KoLeo Differential Entropy
def dino_loss(
    s_logits: torch.Tensor,
    t_logits: torch.Tensor,
    t_center: torch.Tensor,
    tau_s: float = 0.1,
    tau_t: float = 0.05,
) -> torch.Tensor:
    p_t = F.softmax((t_logits - t_center) / tau_t, dim=-1)
    log_p_s = F.log_softmax(s_logits / tau_s, dim=-1)
    return (-(p_t * log_p_s).sum(dim=-1)).mean().unsqueeze(0)

def koleo_loss(z: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    b = z.shape[0]
    z_norm = z / (torch.norm(z, dim=-1, keepdim=True) + 1e-12)
    dist_matrix = torch.cdist(z_norm, z_norm, p=2)
    diag_mask = torch.eye(b, device=z.device) * 1e9
    min_dist, _ = (dist_matrix + diag_mask).min(dim=-1)
    return (-torch.log(min_dist + eps)).mean().unsqueeze(0)

dino_calc = dino_loss(golden["s_logits"], golden["t_logits"], golden["t_center"])
diff_dino = torch.abs(dino_calc - golden["dino_loss_ref"]).item()
assert diff_dino < 1e-5

koleo_calc = koleo_loss(golden["koleo_z"])
diff_koleo = torch.abs(koleo_calc - golden["koleo_loss_ref"]).item()
assert diff_koleo < 1e-5

In [ ]:
# 4. MAE Visible Patch Gathering, I-JEPA Feature Prediction & Anti-Pattern Check
def mae_patch_gather(patches: torch.Tensor, indices: torch.Tensor) -> torch.Tensor:
    b = patches.shape[0]
    return torch.stack([patches[i, indices[i]] for i in range(b)], dim=0)

def jepa_feature_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return F.mse_loss(pred, target).unsqueeze(0)

mae_calc = mae_patch_gather(golden["mae_patches"], golden["mae_indices"])
diff_mae = torch.max(torch.abs(mae_calc - golden["mae_gathered_ref"])).item()
assert diff_mae < 1e-5

jepa_calc = jepa_feature_loss(golden["jepa_pred"], golden["jepa_target"])
diff_jepa = torch.abs(jepa_calc - golden["jepa_loss_ref"]).item()
assert diff_jepa < 1e-5

# Misuse Anti-Pattern: Passing unnormalized vectors to BYOL causes scale explosion
unnorm_p = golden["byol_p"] * 1000.0
unnorm_z = golden["byol_z"] * 1000.0
# The correct normalized function remains strictly invariant to scaling:
inv_check = byol_prediction_loss(unnorm_p, unnorm_z)
assert torch.abs(inv_check - byol_calc).item() < 1e-5, "Failure: BYOL loss must be scale-invariant"

print("=" * 65)
print(f"{'Operator':<32} | {'Max Abs Error':<15} | Status")
print("-" * 65)
print(f"{'BYOL Cosine Prediction Loss':<32} | {diff_byol:<15.2e} | PASSED")
print(f"{'VICReg Regularization Loss':<32} | {diff_vicreg:<15.2e} | PASSED")
print(f"{'DINO Soft Distillation Loss':<32} | {diff_dino:<15.2e} | PASSED")
print(f"{'KoLeo Differential Entropy':<32} | {diff_koleo:<15.2e} | PASSED")
print(f"{'MAE Patch Gather':<32} | {diff_mae:<15.2e} | PASSED")
print(f"{'I-JEPA Latent MSE Loss':<32} | {diff_jepa:<15.2e} | PASSED")
print("=" * 65)

## 8. Summary, Key Takeaways & Cross-Domain Generalisation

- [x] **BYOL/SimSiam**: Asymmetry and stop-gradients prevent representation collapse without contrastive negative pairs.
- [x] **VICReg**: Direct statistical constraints on embedding variance ($\gamma=1.0$) and off-diagonal covariance eliminate dimensional collapse.
- [x] **DINO**: Dynamic centering buffer $c$ prevents dimensional mode collapse, while sharpening ($\tau_t < \tau_s$) prevents uniform collapse.
- [x] **KoLeo**: Kozachenko-Leonenko nearest-neighbor differential entropy estimator maximizes coverage and uniformity on the unit hypersphere.
- [x] **MAE & I-JEPA**: MAE processes only visible patches ($4\times$ speedup) while I-JEPA predicts representations directly in feature space without decoding pixels.

**Cross-Domain Generalisation**:
Non-contrastive self-supervised techniques directly empower foundation models across vision (DINOv2, MAE), audio (data2vec), robotics (JEPA world models), and genomics (masked biological sequence representation).

See curriculum index: [INDEX_BY_GOAL](../../../INDEX_BY_GOAL.md).